# T03 — Integración de un modelo con una aplicación
## Detector de comentarios tóxicos (BERT) — Civil Comments

**Equipo:** Branco Merino y Matías Díaz · **Paralelo:** D03
**Asignatura:** Desarrollo de Soluciones Inteligentes · Taller Evaluado 03 (23-10-2026)

**Problema y usuario.** Un moderador de una plataforma de comentarios escribe o pega un comentario y la aplicación le indica si es potencialmente tóxico, con la probabilidad asociada.

**Base utilizada (ruta "modelo previamente trabajado").** Modelo del Taller 04: `google-bert/bert-base-uncased` ajustado (fine-tuning) sobre una muestra de Civil Comments (`google/civil_comments`, https://huggingface.co/datasets/google/civil_comments). Etiqueta: `toxico = 1` si `toxicity >= 0.5`.

**Qué se modificó respecto del T04.** No se entrena ni se ajusta nada. Se agregan: (1) carga del modelo guardado desde un `.zip`, (2) contrato de entrada, (3) validación con errores y advertencias, (4) función de inferencia que devuelve un resultado interpretable, (5) casos válido/inválido y verificación de que los pesos no cambian.

## 0. Instrucciones de ejecución
1. Abrir en Google Colab (la GPU es opcional; con CPU funciona, solo es más lento).
2. Tener a mano `modelos_toxicidad.zip` (generado en el T04; contiene la carpeta `modelo_bert_toxicidad/` con `config.json`, `model.safetensors` y los archivos del tokenizador).
3. Ejecutar todas las celdas en orden (**Entorno → Ejecutar todo**). Si el ZIP no está en `/content`, la celda 2 abrirá un selector para subirlo.
4. Dependencias: `torch`, `transformers`, `numpy`, `matplotlib` (se instalan en la celda 1).

No se usa API ni nuevo entrenamiento. La interfaz web (Vigía, sección 9) es solo una capa para probar el modelo ya cargado.

## 1. Dependencias y configuración

In [1]:
!pip -q install transformers torch accelerate

import os, glob, zipfile, re, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ZIP_NOMBRE = "modelo_bert_toxicidad.zip"    # archivo con los modelos del T04
DIR_EXTRACCION = "/content/modelo_bert"
MAX_LENGTH = 256                        # mismo valor usado en el entrenamiento
UMBRAL = 0.5                            # probabilidad >= 0.5 => "Tóxico" (equivale al argmax)
MAX_CARACTERES = 5000                   # límite de entrada aceptada
MIN_CARACTERES = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

Dispositivo: cuda


## 2. Cargar el modelo BERT desde el archivo ZIP
Se descomprime el ZIP, se localiza la carpeta que contiene `config.json` junto con los pesos y se cargan **modelo y tokenizador una sola vez**. El modelo queda en modo evaluación (`eval()`): la inferencia no vuelve a ajustar nada.

In [ ]:
# 2.1 Ubicar (o pedir) el ZIP
ruta_zip = os.path.join("/content", ZIP_NOMBRE)
if not os.path.exists(ruta_zip):
    zips = glob.glob("/content/*.zip")
    if zips:
        ruta_zip = zips[0]
    else:
        try:
            from google.colab import files
            subidos = files.upload()                  # seleccionar modelos_toxicidad.zip
            ruta_zip = os.path.join("/content", list(subidos.keys())[0])
        except ImportError:
            raise FileNotFoundError(f"No se encontró {ZIP_NOMBRE}. Colócalo en esta carpeta.")
print("ZIP utilizado:", ruta_zip)

# 2.2 Descomprimir
os.makedirs(DIR_EXTRACCION, exist_ok=True)
with zipfile.ZipFile(ruta_zip) as z:
    z.extractall(DIR_EXTRACCION)

# 2.3 Localizar la carpeta del modelo BERT (config.json + pesos)
RUTA_BERT = None
for raiz, _, archivos in os.walk(DIR_EXTRACCION):
    tiene_pesos = any(a in ("model.safetensors", "pytorch_model.bin") for a in archivos)
    if "config.json" in archivos and tiene_pesos:
        RUTA_BERT = raiz
        break
if RUTA_BERT is None:
    raise FileNotFoundError("El ZIP no contiene una carpeta de modelo con config.json y pesos.")
print("Carpeta del modelo:", RUTA_BERT)

# 2.4 Cargar tokenizador y modelo UNA sola vez
tokenizer = AutoTokenizer.from_pretrained(RUTA_BERT)
modelo = AutoModelForSequenceClassification.from_pretrained(RUTA_BERT).to(DEVICE)
modelo.eval()                                       # modo inferencia: sin dropout ni reajuste
for p in modelo.parameters():
    p.requires_grad_(False)

print("Etiquetas del modelo:", modelo.config.id2label)
assert modelo.config.num_labels == 2, "Se esperaba un clasificador binario"

## 3. Contrato de entrada y salida

**Entrada** (un comentario individual):

| Campo | Significado | Tipo | Unidad / formato | Obligatorio | Regla |
|---|---|---|---|---|---|
| `texto` | Comentario a evaluar | `str` | Texto libre en **inglés** (único idioma con fiabilidad) | Sí | Entre 3 y 5.000 caracteres, no vacío ni solo espacios, con al menos una letra |

**Salida** (diccionario):

| Campo | Tipo | Significado |
|---|---|---|
| `etiqueta` | `str` | `"Tóxico"` o `"No tóxico"` |
| `prob_toxico` | `float` ∈ [0, 1] | Probabilidad que asigna el modelo a la clase tóxica |
| `confianza` | `float` ∈ [0.5, 1] | Probabilidad de la clase predicha |
| `umbral` | `float` | Umbral de decisión (0,5) |
| `advertencias` | `list[str]` | Avisos que **no** impiden ejecutar (ver abajo) |
| `limitacion` | `str` | Limitación relevante para interpretar el resultado |

**Error vs. advertencia.** Un *error* (`ErrorEntrada`) detiene la ejecución porque el modelo no puede o no debe procesar esa entrada. Una *advertencia* permite ejecutar, pero avisa que el resultado puede ser menos fiable:
- Texto que parece estar en español (u otro idioma distinto del inglés) → el modelo solo es fiable en inglés (Civil Comments); en cualquier otro idioma su fiabilidad es nula.
- Texto más largo que 256 *tokens* → se trunca, igual que en el entrenamiento (referencia: `MAX_LENGTH = 256`).

## 4. Validación de entradas

In [ ]:
class ErrorEntrada(ValueError):
    """Entrada inválida: se detiene antes de ejecutar el modelo."""

PALABRAS_ES = {"el","la","los","las","de","del","que","y","en","un","una","por","con",
               "para","se","muy","pero","esto","eres","está","como","qué","tú","eso","lo"}

def validar_entrada(texto):
    """Devuelve (texto_limpio, advertencias). Lanza ErrorEntrada si es inválida."""
    if texto is None:
        raise ErrorEntrada("Falta el campo obligatorio 'texto'.")
    if not isinstance(texto, str):
        raise ErrorEntrada(f"'texto' debe ser str, pero se recibió {type(texto).__name__}.")
    limpio = texto.strip()
    if limpio == "":
        raise ErrorEntrada("'texto' está vacío o contiene solo espacios.")
    if len(limpio) < MIN_CARACTERES:
        raise ErrorEntrada(f"'texto' es demasiado corto ({len(limpio)} caracteres; mínimo {MIN_CARACTERES}).")
    if len(limpio) > MAX_CARACTERES:
        raise ErrorEntrada(f"'texto' excede el máximo de {MAX_CARACTERES} caracteres ({len(limpio)}).")
    if not re.search(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]", limpio):
        raise ErrorEntrada("'texto' no contiene letras (solo números o símbolos).")

    advertencias = []
    palabras = re.findall(r"\w+", limpio.lower())
    if sum(p in PALABRAS_ES for p in palabras) >= 2:
        advertencias.append("El texto parece estar en español; el modelo solo es fiable en inglés. "
                            "En cualquier otro idioma su fiabilidad es nula.")
    n_tokens = len(tokenizer(limpio, truncation=False, verbose=False)["input_ids"])
    if n_tokens > MAX_LENGTH:
        advertencias.append(f"El texto tiene {n_tokens} tokens y se truncará a {MAX_LENGTH}; "
                            "se ignorará el final del comentario.")
    return limpio, advertencias

## 5. Función de inferencia (sin reajustar el modelo)

In [ ]:
LIMITACION = ("Modelo entrenado con Civil Comments (principalmente inglés, etiqueta binaria con umbral 0,5 "
              "sobre una puntuación continua). Solo es fiable en inglés (en otros idiomas su fiabilidad es nula). Puede errar en ironía o contexto, y no sustituye "
              "la revisión humana.")

ID_TOXICO = 1   # índice de la clase "Tóxico" (id2label del modelo)

@torch.no_grad()
def predecir(texto):
    texto_limpio, advertencias = validar_entrada(texto)         # 1) validar
    enc = tokenizer(texto_limpio, truncation=True, max_length=MAX_LENGTH,
                    return_tensors="pt").to(DEVICE)              # 2) preprocesar (tokenizador del T04)
    logits = modelo(**enc).logits                                # 3) modelo ya entrenado
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
    p_tox = float(probs[ID_TOXICO])
    es_toxico = p_tox >= UMBRAL
    return {                                                     # 4) salida interpretable
        "etiqueta": "Tóxico" if es_toxico else "No tóxico",
        "prob_toxico": round(p_tox, 4),
        "confianza": round(p_tox if es_toxico else 1 - p_tox, 4),
        "umbral": UMBRAL,
        "advertencias": advertencias,
        "limitacion": LIMITACION,
    }

def mostrar(texto):
    print("ENTRADA :", repr(texto))
    try:
        r = predecir(texto)
    except ErrorEntrada as e:
        print("ERROR DE ENTRADA ->", e)
        print("(el modelo NO se ejecutó)\n")
        return None
    print(f"SALIDA  : {r['etiqueta']}  (prob. tóxico = {r['prob_toxico']:.2%}, umbral = {r['umbral']})")
    for a in r["advertencias"]:
        print("ADVERTENCIA ->", a)
    print("LIMITACIÓN:", r["limitacion"], "\n")
    return r

## 6. Demostración

### 6.1 Casos válidos

In [ ]:
print("=== CASO VÁLIDO 1 (comentario neutro) ===")
r1 = mostrar("I respectfully disagree with the council's decision, but I appreciate the open discussion.")

print("=== CASO VÁLIDO 2 (comentario ofensivo) ===")
r2 = mostrar("You are a complete idiot and nobody wants you here.")

### 6.2 Casos inválidos (error: el modelo no se ejecuta)

In [ ]:
print("=== CASO INVÁLIDO 1 (texto vacío) ===")
mostrar("    ")

print("=== CASO INVÁLIDO 2 (tipo incorrecto) ===")
mostrar(12345)

print("=== CASO INVÁLIDO 3 (campo faltante) ===")
mostrar(None)

print("=== CASO INVÁLIDO 4 (sin letras) ===")
mostrar("!!! 123 ???")

### 6.3 Caso con advertencia (se ejecuta, pero se avisa; no es un error)

In [ ]:
print("=== CASO CON ADVERTENCIA (español) ===")
mostrar("Esto es muy malo y no me gusta para nada lo que dices en este comentario.")

### 6.4 Probar un comentario propio
Edita `TEXTO_USUARIO` y ejecuta la celda.

In [ ]:
TEXTO_USUARIO = "Thanks for the article, it was really helpful."
_ = mostrar(TEXTO_USUARIO)

## 7. Control: la inferencia no reajusta el modelo
Se compara una suma de verificación de todos los pesos antes y después de varias predicciones, y se confirma que el modelo está en modo evaluación.

In [ ]:
def checksum_pesos():
    return float(sum(p.detach().double().sum().item() for p in modelo.parameters()))

antes = checksum_pesos()
for t in ["Have a nice day!", "You are pathetic.", "The meeting is at noon."]:
    predecir(t)
despues = checksum_pesos()

print("Suma de pesos antes  :", antes)
print("Suma de pesos después:", despues)
assert antes == despues, "Los pesos cambiaron: la inferencia no debe reajustar el modelo"
assert not modelo.training, "El modelo debe estar en modo eval()"
assert all(not p.requires_grad for p in modelo.parameters())
print("Control superado: pesos intactos, modo eval, sin gradientes.")

## 8. Diagrama de arquitectura

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.6))
ax.axis("off"); ax.set_xlim(0, 13); ax.set_ylim(0, 3.6)

cajas = [
    ("Entrada\ntexto (str)", 0.2, "#dbeafe"),
    ("Validación\nvacío / tipo / longitud\n/ letras", 2.7, "#fde68a"),
    ("Tokenizador BERT\n(del T04, max 256)", 5.2, "#e9d5ff"),
    ("BERT fine-tuned\n(cargado desde ZIP,\neval, sin reajuste)", 7.7, "#bbf7d0"),
    ("Salida\netiqueta + prob.\n+ advertencias\n+ limitación", 10.4, "#fecaca"),
]
for texto, x, color in cajas:
    ax.add_patch(plt.Rectangle((x, 1.2), 2.3, 1.6, fc=color, ec="black", lw=1.3))
    ax.text(x + 1.15, 2.0, texto, ha="center", va="center", fontsize=9)
for i in range(len(cajas) - 1):
    x0 = cajas[i][1] + 2.3
    x1 = cajas[i + 1][1]
    ax.annotate("", xy=(x1, 2.0), xytext=(x0, 2.0), arrowprops=dict(arrowstyle="->", lw=1.6))

# Rama de error
ax.annotate("", xy=(3.85, 0.55), xytext=(3.85, 1.2), arrowprops=dict(arrowstyle="->", lw=1.6, color="red"))
ax.text(3.85, 0.3, "ErrorEntrada: mensaje claro\n(el modelo NO se ejecuta)", ha="center", va="center",
        fontsize=9, color="red")
ax.set_title("Recorrido: entrada → validación → modelo → salida", fontsize=11)
plt.tight_layout()
plt.savefig("diagrama_arquitectura_T03.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Interfaz básica - Vigía

In [ ]:
# Interfaz básica para probar el detector de toxicidad (BERT)
!pip -q install gradio

import gradio as gr

def probar(texto):
    try:
        r = predecir(texto)
    except ErrorEntrada as e:
        # Entrada inválida: el modelo no se ejecuta
        return "ERROR DE ENTRADA", 0.0, f"⚠️ {e}\n(el modelo NO se ejecutó)"

    avisos = "\n".join(f"⚠️ {a}" for a in r["advertencias"]) or "Sin advertencias."
    detalle = (f"Confianza: {r['confianza']:.2%}\n"
               f"Umbral: {r['umbral']}\n\n"
               f"{avisos}\n\n"
               f"Limitación: {r['limitacion']}")
    return r["etiqueta"], r["prob_toxico"], detalle

demo = gr.Interface(
    fn=probar,
    inputs=gr.Textbox(lines=4, label="Comentario a evaluar (solo fiable en inglés)",
                      placeholder="Escribe o pega un comentario..."),
    outputs=[
        gr.Textbox(label="Etiqueta"),
        gr.Number(label="Probabilidad de toxicidad (0 a 1)"),
        gr.Textbox(label="Detalle", lines=8),
    ],
    examples=[
        ["I respectfully disagree with the council's decision, but I appreciate the open discussion."],
        ["You are a complete idiot and nobody wants you here."],
        ["Esto es muy malo y no me gusta para nada lo que dices en este comentario."],
        ["    "],
    ],
    title="Detector de comentarios tóxicos Vigía (BERT)",
    description="Taller T03 · Modelo bert-base-uncased ajustado con Civil Comments. "
                "Ayuda para priorizar revisión; no sustituye la revisión humana.",
    flagging_mode="never",
)

demo.launch(share=False)   # usa share=True para un enlace público

## 10. Decisión, limitación y mejora pendiente

**Decisión de integración.** El modelo y el tokenizador se cargan una sola vez desde el ZIP y se dejan en `eval()` con gradientes desactivados; la función `predecir()` solo tokeniza y calcula la salida. Se usa el mismo `max_length = 256` del entrenamiento para mantener la correspondencia entre entrenamiento e inferencia. Se separó **error** (entrada imposible o inválida → se detiene) de **advertencia** (se ejecuta, pero se avisa: texto en español —el modelo solo es fiable en inglés— o texto truncado).

**Limitación.** El modelo se entrenó con Civil Comments (principalmente inglés) y con una etiqueta binaria obtenida de un umbral 0,5 sobre una puntuación continua; los comentarios cercanos al umbral son ambiguos y el modelo solo es fiable en inglés y en cualquier otro idioma su fiabilidad es nula. El resultado es una ayuda para priorizar revisión, no un veredicto.

**Mejora / prueba pendiente.** Ampliar el soporte a otros idiomas (por ejemplo, reentrenando con un corpus en español o multilingüe) y calibrar el umbral con un conjunto de validación según el costo de falsos positivos vs. falsos negativos.

## 11. Reutilización y uso de IA
- **Base reutilizada:** notebook del Taller 04 (pipeline de preparación, TF-IDF + Regresión Logística y BERT), dataset Civil Comments (Hugging Face) y `bert-base-uncased`.
- **Cambios propios:** contrato de entrada/salida, validación con errores y advertencias, función de inferencia, carga desde ZIP, control de pesos intactos, diagrama.
- **IA generativa:** se utilizó Claude (Anthropic) para estructurar y redactar funciones del código de este notebook. El equipo ejecutó el notebook, revisó cada celda y verificó la salida de los casos válido, inválido y con advertencia.